In [1]:
from platform import python_version
python_version()
import numpy as np
import torch
import numpy as np
from arsf_envi_reader import envi_header
import shutil
import os

import json
import math
import affine
import pandas as pd
import numpy as np
# import matplotlib.pyplot as plt
# import matplotlib.gridspec as gridspec
from osgeo import gdal,ogr,osr

import numpy as np
# import matplotlib.pyplot as plt
from scipy.optimize import curve_fit



from tqdm import tqdm
# import multiprocess as mp
from numpy import trapz

C:\Users\laral\AppData\Roaming\Python\Python39\site-packages\pandas\core\arrays\masked.py:60: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


In [2]:
# ============================================================
# USER SETTINGS (change these two paths)
# ============================================================
hdr_path   = r"D:\wenqu\aviris\site2a_aviris_ng_srf_data\ang20190704t193319rfl\ang20190704t193319_rfl_v2v2_img.hdr"
csv_path = r"D:\wenqu\2024_data\SVC\update1\site3\site3.csv"

out_path   = r"D:\wenqu\2024_data\SVC\update1\site3\site3_convoluted.csv"

In [3]:
# ============================================================
# 1) Your Gaussian SRF (use exactly this)
# ============================================================
def gauss(x, fwhm, center):
    sigma = fwhm / (2 * np.sqrt(2 * np.log(2)))
    y = np.exp(-((x - center) ** 2) / (2 * sigma ** 2))
    return y

In [4]:
# ============================================================
# 2) Read AVIRIS wavelength + FWHM from ENVI .hdr (no extra libs)
# ============================================================
hdr_txt = open(hdr_path, "r", encoding="utf-8", errors="ignore").read()

# --- wavelength block ---
w_start = hdr_txt.lower().find("wavelength")
if w_start == -1:
    raise ValueError("Cannot find 'wavelength' in header")

w_eq = hdr_txt.find("=", w_start)
w_lb = hdr_txt.find("{", w_eq)
w_rb = hdr_txt.find("}", w_lb)
if w_eq == -1 or w_lb == -1 or w_rb == -1:
    raise ValueError("Cannot parse wavelength {...} block in header")

w_str = hdr_txt[w_lb + 1 : w_rb].replace("\n", " ").replace("\r", " ").strip()
av_center = np.array([float(x) for x in w_str.split(",") if x.strip() != ""], dtype=float)

# --- fwhm block ---
f_start = hdr_txt.lower().find("fwhm")
if f_start == -1:
    raise ValueError("Cannot find 'fwhm' in header")

f_eq = hdr_txt.find("=", f_start)
f_lb = hdr_txt.find("{", f_eq)
f_rb = hdr_txt.find("}", f_lb)
if f_eq == -1 or f_lb == -1 or f_rb == -1:
    raise ValueError("Cannot parse fwhm {...} block in header")

f_str = hdr_txt[f_lb + 1 : f_rb].replace("\n", " ").replace("\r", " ").strip()
av_fwhm = np.array([float(x) for x in f_str.split(",") if x.strip() != ""], dtype=float)

if len(av_center) != len(av_fwhm):
    raise ValueError(f"Header mismatch: len(wavelength)={len(av_center)} vs len(fwhm)={len(av_fwhm)}")

# sort AVIRIS by center wavelength (good practice)
k = np.argsort(av_center)
av_center = av_center[k]
av_fwhm = av_fwhm[k]

print("AVIRIS bands:", len(av_center), "  range:", av_center.min(), "-", av_center.max())

AVIRIS bands: 425   range: 376.719576 - 2500.399576


In [5]:
# ============================================================
# 3) Read SVC CSV (col1 = wavelength, rest = reflectance columns)
# ============================================================
df = pd.read_csv(csv_path)

wl_col = df.columns[0]
svc_wl = df.iloc[:, 0].astype(float).to_numpy()

spec_cols = list(df.columns[1:])
svc_R = df.iloc[:, 1:].apply(pd.to_numeric, errors="coerce").to_numpy(dtype=float)

# sort by wavelength
idx = np.argsort(svc_wl)
svc_wl = svc_wl[idx]
svc_R = svc_R[idx, :]

# clean invalid wavelength rows
m = np.isfinite(svc_wl)
svc_wl = svc_wl[m]
svc_R = svc_R[m, :]

print("SVC wl:", svc_wl.shape, "  spectra:", svc_R.shape)


SVC wl: (1024,)   spectra: (1024, 22)


In [6]:
# ============================================================
# 4) Convolution (NO window limit; use full SVC wavelength grid)
#    For each AVIRIS band:
#      SRF_b(λ) = Gaussian(center=av_center[b], fwhm=av_fwhm[b]) on svc_wl
#      R_b = ∫ R(λ)*SRF_b(λ) dλ / ∫ SRF_b(λ) dλ
# ============================================================
n_bands = len(av_center)
n_spec  = svc_R.shape[1]

conv = np.full((n_bands, n_spec), np.nan, dtype=float)

for i in range(n_bands):
    center = av_center[i]
    fwhm   = av_fwhm[i]

    # SRF on FULL SVC wavelength grid (no limit)
    srf = gauss(svc_wl, fwhm, center)

    # denom = ∫ SRF dλ
    denom = np.trapz(srf, svc_wl)
    if not np.isfinite(denom) or denom <= 0:
        continue

    # numerator = ∫ R(λ)*SRF(λ) dλ  (vectorized over spectra columns)
    num = np.trapz(svc_R * srf[:, None], svc_wl, axis=0)

    conv[i, :] = num / denom

print("Convolved shape:", conv.shape)

Convolved shape: (425, 22)


In [7]:
# ============================================================
# 5) Save output: rows = AVIRIS band centers, columns = your spectra
# ============================================================
out_df = pd.DataFrame(conv, columns=spec_cols)
out_df.insert(0, "aviris_center_wavelength", av_center)
out_df.to_csv(out_path, index=False)

print("DONE. Saved:", out_path)

DONE. Saved: D:\wenqu\2024_data\SVC\update1\site3\site3_convoluted.csv
